# E3: Spatial + DCT CONCAT — training only
Primary independent initialization. Validation selects checkpoint and locks threshold. Test belongs to the separate notebook. Frozen protocol: see E3_spatial_frequency_concat_protocol.md.

In [ ]:
%pip install -q albumentations onnx onnxscript onnxruntime scikit-learn pandas psutil

In [ ]:
import os, glob, json, time, math, random, hashlib, shutil, sys, platform
from pathlib import Path, PurePosixPath
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import onnxruntime as ort
import psutil
SAMPLE_SEED = 42
INPUT_SIZE = 224
FREQ_DIM = 64
BBOX_JITTER_P = 0.20
BBOX_JITTER_SCALE = (0.95, 1.05)
BBOX_JITTER_TRANSLATE = 0.05
DCT_EPS = 1e-6
CELEBA_MEAN = [0.5931, 0.4690, 0.4229]
CELEBA_STD = [0.2471, 0.2214, 0.2157]
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
print({'python':sys.version, 'torch':torch.__version__, 'opencv':cv2.__version__, 'ort':ort.__version__, 'albumentations':A.__version__})

## Paths and run mode
Enable Kaggle GPU and Internet for ImageNet weights. Set DATA_ROOT to the directory containing Data/ and metas/. E1 config + exact NPZ/cache must be attached.

In [ ]:
E1_RUN_MODE = "preliminary"
E1_ARTIFACT_DIR = ""
DATA_ROOT = "/kaggle/input/EDIT_ME/CelebA_Spoof"
INIT_MODE = "independent"
NUM_WORKERS = 4
SOURCE_NOTEBOOK_HASHES = {'antispoof/notebooks/e1_spatial_v5_3/e1_v5_3_training.ipynb': '971d4bbf0ff709ab073b0a7266f26132cf7e23cc05ad73bf08f86d3ff509d539', 'antispoof/notebooks/e2_frequency_dct/e2_frequency_only_dct_train.ipynb': '5dacdb5386e18209455c8c51594bf066289e2f85b1f23fc3473a681e5de3612c'}
FROZEN_CLASS_COUNTS_PATH = ""  # official only if E1 config lacks class_counts

CHECKPOINT_INPUT_PATH = ""  # optional trusted E3 checkpoint_last.pth from the same protocol


In [ ]:
def find_exact_artifact(filename, explicit_dir=""):
    if explicit_dir:
        if not explicit_dir.startswith("/"):
            explicit_dir = "/" + explicit_dir
        p = os.path.join(explicit_dir, filename)
        if os.path.exists(p):
            return p
        # Tự động quét tìm trong các thư mục con data_protocol, deployment, metadata
        sub_matches = sorted(set(glob.glob(os.path.join(explicit_dir, "**", filename), recursive=True)))
        if len(sub_matches) == 1:
            return sub_matches[0]
        if len(sub_matches) > 1:
            raise RuntimeError(f"Multiple copies of {filename} found under {explicit_dir}:\n" + "\n".join(sub_matches))
        raise FileNotFoundError(f"Missing required artifact {filename} under {explicit_dir}")

    matches = glob.glob(os.path.join("/kaggle/input", "**", filename), recursive=True)
    matches = sorted(set(matches))
    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input.")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple copies of {filename} found:\n" + "\n".join(matches))
    return matches[0]

def find_optional_artifact(filename, explicit_dir=""):
    try:
        return find_exact_artifact(filename, explicit_dir)
    except FileNotFoundError:
        return None

def label_of(meta_value):
    if len(meta_value) <= 40:
        raise ValueError(f"Metadata entry too short: len={len(meta_value)}")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1, 2, 3, 4, 5, 6, 7]:
        return 1
    return 2

def class_counts(keys, meta):
    y = np.array([label_of(meta[str(k)]) for k in keys], dtype=np.int64)
    return {c: int((y == c).sum()) for c in [0, 1, 2]}

def split_fingerprint(keys):
    payload = "\n".join(sorted(map(str, keys))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def selected_cache_fingerprint(keys, records):
    h = hashlib.sha256()
    for key in sorted(map(str, keys)):
        h.update(key.encode("utf-8"))
        h.update(b"\0")
        h.update(
            json.dumps(
                records[key],
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(b"\n")
    return h.hexdigest()

def runtime_crop_bgr(img, bbox_xyxy, bbox_expansion_factor):
    original_height, original_width = img.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")

    max_dim = max(w, h)
    center_x = x1 + w / 2.0
    center_y = y1 + h / 2.0

    x_start = int(center_x - max_dim * bbox_expansion_factor / 2.0)
    y_start = int(center_y - max_dim * bbox_expansion_factor / 2.0)
    crop_size = int(max_dim * bbox_expansion_factor)
    if crop_size <= 0:
        raise ValueError(f"Invalid crop_size={crop_size}")

    crop_x1 = max(0, x_start)
    crop_y1 = max(0, y_start)
    crop_x2 = min(original_width, x_start + crop_size)
    crop_y2 = min(original_height, y_start + crop_size)

    top_pad = int(max(0, -y_start))
    left_pad = int(max(0, -x_start))
    bottom_pad = int(max(0, (y_start + crop_size) - original_height))
    right_pad = int(max(0, (x_start + crop_size) - original_width))

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        raise RuntimeError(f"Empty crop intersection for bbox={bbox_xyxy}")

    cropped_img = img[crop_y1:crop_y2, crop_x1:crop_x2, :]
    result = cv2.copyMakeBorder(
        cropped_img,
        top_pad,
        bottom_pad,
        left_pad,
        right_pad,
        cv2.BORDER_REFLECT_101,
    )

    if result.shape[0] != crop_size or result.shape[1] != crop_size:
        result = cv2.resize(
            result,
            (crop_size, crop_size),
            interpolation=cv2.INTER_AREA,
        )
    return result

def read_cached_face_crop_rgb(
    img_path,
    cache_record,
    bbox_jitter=False,
    bbox_jitter_p=BBOX_JITTER_P,
    jitter_scale=BBOX_JITTER_SCALE,
    jitter_translate=BBOX_JITTER_TRANSLATE,
):
    image_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {img_path}")

    if cache_record.get("status") == "INVALID":
        raise RuntimeError(
            f"Invalid cached preprocessing record for {img_path}: {cache_record}"
        )

    x1, y1, x2, y2 = map(float, cache_record["bbox_xyxy"])
    w = x2 - x1
    h = y2 - y1
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive cached bbox for {img_path}")

    if bbox_jitter and random.random() < float(bbox_jitter_p):
        base = max(w, h)
        cx = x1 + w / 2.0
        cy = y1 + h / 2.0

        scale = random.uniform(
            float(jitter_scale[0]),
            float(jitter_scale[1]),
        )
        cx += random.uniform(-jitter_translate, jitter_translate) * base
        cy += random.uniform(-jitter_translate, jitter_translate) * base
        w *= scale
        h *= scale

        x1 = cx - w / 2.0
        y1 = cy - h / 2.0
        x2 = x1 + w
        y2 = y1 + h

    crop_bgr = runtime_crop_bgr(
        image_bgr,
        (x1, y1, x2, y2),
        bbox_expansion_factor=float(cache_record["crop_factor"]),
    )
    return cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

def resize_rgb_to_input(img_rgb, size=INPUT_SIZE):
    old_h, old_w = img_rgb.shape[:2]
    ratio = float(size) / max(old_h, old_w)
    scaled_h = max(1, int(old_h * ratio))
    scaled_w = max(1, int(old_w * ratio))
    interpolation = cv2.INTER_LANCZOS4 if ratio > 1.0 else cv2.INTER_AREA

    img = cv2.resize(
        img_rgb,
        (scaled_w, scaled_h),
        interpolation=interpolation,
    )

    delta_w = size - scaled_w
    delta_h = size - scaled_h
    top, bottom = delta_h // 2, delta_h - delta_h // 2
    left, right = delta_w // 2, delta_w - delta_w // 2

    return cv2.copyMakeBorder(
        img,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_REFLECT_101,
    )

def rgb_to_luminance_float(img_rgb):
    rgb = img_rgb.astype(np.float32) / 255.0
    # Standard RGB luminance weighting.
    return (
        0.299 * rgb[..., 0]
        + 0.587 * rgb[..., 1]
        + 0.114 * rgb[..., 2]
    ).astype(np.float32)

def luminance_to_dct_map(luma):
    if luma.shape != (INPUT_SIZE, INPUT_SIZE):
        raise ValueError(f"Expected {(INPUT_SIZE, INPUT_SIZE)}, got {luma.shape}")

    coeff = cv2.dct(np.ascontiguousarray(luma, dtype=np.float32))
    coeff = np.sign(coeff) * np.log1p(np.abs(coeff))

    mean = float(coeff.mean())
    std = float(coeff.std())
    coeff = (coeff - mean) / max(std, DCT_EPS)

    return coeff[None, ...].astype(np.float32)

def rgb_to_frequency_tensor(img_rgb):
    img_rgb = resize_rgb_to_input(img_rgb, INPUT_SIZE)
    luma = rgb_to_luminance_float(img_rgb)
    dct_map = luminance_to_dct_map(luma)
    return torch.from_numpy(np.ascontiguousarray(dct_map))

def runtime_preprocess_rgb(img_rgb, model_img_size, mean, std):
    """Match runtime resize + REFLECT_101 letterbox + normalize + NCHW."""
    if img_rgb is None or img_rgb.size == 0:
        raise ValueError("Cannot preprocess an empty image")

    new_size = int(model_img_size)
    old_size = img_rgb.shape[:2]
    ratio = float(new_size) / max(old_size)
    scaled_shape = tuple(max(1, int(x * ratio)) for x in old_size)
    interpolation = cv2.INTER_LANCZOS4 if ratio > 1.0 else cv2.INTER_AREA
    img = cv2.resize(img_rgb, (scaled_shape[1], scaled_shape[0]), interpolation=interpolation)

    delta_w = new_size - scaled_shape[1]
    delta_h = new_size - scaled_shape[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT_101)

    arr = img.transpose(2, 0, 1).astype(np.float32) / 255.0
    if mean is not None and std is not None:
        mean_arr = np.asarray(mean, dtype=np.float32).reshape(3, 1, 1)
        std_arr = np.asarray(std, dtype=np.float32).reshape(3, 1, 1)
        arr = (arr - mean_arr) / std_arr
    return torch.from_numpy(np.ascontiguousarray(arr))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def json_clean(x):
    if isinstance(x, dict): return {str(k):json_clean(v) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return [json_clean(v) for v in x]
    if isinstance(x, np.ndarray): return json_clean(x.tolist())
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (float,np.floating)): return float(x) if np.isfinite(x) else None
    return x

def save_json(path, value):
    Path(path).write_text(json.dumps(json_clean(value),indent=2,allow_nan=False))

def branch_inputs(crop_rgb):
    # One shared augmented crop; exact E1 spatial and E2 DCT functions.
    rgb = runtime_preprocess_rgb(crop_rgb, 224, CELEBA_MEAN, CELEBA_STD)
    freq = rgb_to_frequency_tensor(crop_rgb)
    return rgb, freq

def verify_disjoint(splits):
    sets = [set(map(str,k)) for k in splits]
    if any(len(s)!=len(k) for s,k in zip(sets,splits)): raise RuntimeError('Duplicate split keys')
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise RuntimeError('Train/Val/Test overlap')
    subjects = [{PurePosixPath(k).parts[2] for k in s} for s in sets]
    if any(subjects[i]&subjects[j] for i in range(3) for j in range(i+1,3)): raise RuntimeError('Subject overlap')

DCT_SPEC = {'implementation':'cv2.dct float32 orthonormal 2D', 'luminance_weights_rgb':[0.299,0.587,0.114],
 'rgb_range':'uint8 -> float32 / 255', 'dynamic_range':'sign(C)*log1p(abs(C))',
 'normalization':'per-sample z-score over full map, population std', 'epsilon':1e-6, 'mask':'none',
 'resize':'longest side 224; LANCZOS4 upscale, AREA downscale; REFLECT_101 letterbox'}


def binary_pad_score_from_logits(logits_np):
    logits = np.asarray(logits_np, dtype=np.float64)
    real = logits[:, 0]
    spoof = logits[:, 1:]

    m = np.max(spoof, axis=1)
    spoof_lse = m + np.log(
        np.exp(spoof[:, 0] - m)
        + np.exp(spoof[:, 1] - m)
    )
    return real - spoof_lse

def metrics_from_scores(y3, d, threshold):
    y3 = np.asarray(y3, dtype=np.int64)
    d = np.asarray(d, dtype=np.float64)

    is_real = (y3 == 0)
    is_attack = ~is_real
    pred_real = d >= threshold

    bpcer = (
        float(np.mean(~pred_real[is_real]))
        if np.any(is_real) else float("nan")
    )
    apcer = (
        float(np.mean(pred_real[is_attack]))
        if np.any(is_attack) else float("nan")
    )
    acer = 0.5 * (apcer + bpcer)

    y_binary = is_real.astype(np.int64)
    auc = float(roc_auc_score(y_binary, d)) if np.unique(y_binary).size==2 else float('nan')

    accuracy = float(np.mean(pred_real == is_real))
    cm = confusion_matrix(
        y_binary,
        pred_real.astype(np.int64),
        labels=[0, 1],
    ).tolist()

    return {
        "Accuracy": accuracy,
        "APCER": apcer,
        "BPCER": bpcer,
        "ACER": acer,
        "AUC": auc,
        "confusion_matrix": cm,
    }

def sigmoid(x):
    return float(1/(1+np.exp(-np.clip(x,-700,700))))

def pad_metrics(y, d, threshold):
    y,d = np.asarray(y),np.asarray(d,dtype=float)
    if len(y)==0:
        return {'N':0,'N_real':0,'N_attack':0,'AUC':None,'HTER':None,'APCER':None,'BPCER':None,'ACER':None,'Accuracy':None}
    if len(y)!=len(d) or not np.isfinite(d).all(): raise RuntimeError('Invalid scores')
    m=metrics_from_scores(y,d,threshold)
    m.update(N=len(y),N_real=int((y==0).sum()),N_attack=int((y!=0).sum()),HTER=m['ACER'])
    return json_clean(m)

FACE_BINS = [-np.inf,48,100,140,180,np.inf]
FACE_LABELS = ['<48','48–99','100–139','140–179','>=180']
def face_size_breakdown(df, threshold):
    rows=[]
    bins=pd.cut(df.face_min_side, FACE_BINS, labels=FACE_LABELS, right=False)
    for name in FACE_LABELS:
        g=df[bins==name]
        rows.append({'face_size_bin':name,**pad_metrics(g.true_class,g.d,threshold)})
    return pd.DataFrame(rows)

def save_plots(df, threshold, prefix):
    fig,ax=plt.subplots()
    for label,name in [(0,'Real'),(1,'Attack')]:
        vals=df.loc[(df.true_class!=0).astype(int)==label,'d']
        if len(vals): ax.hist(vals,bins=40,alpha=.5,label=name)
    ax.axvline(threshold,linestyle='--',label='Source Validation threshold');ax.legend();ax.set_xlabel('d')
    fig.savefig(str(prefix)+'_score_distribution.png',bbox_inches='tight');plt.close(fig)
    fig,ax=plt.subplots()
    if df.true_class.eq(0).any() and df.true_class.ne(0).any():
        fpr,tpr,_=roc_curve(df.true_class.eq(0).astype(int),df.d);ax.plot(fpr,tpr)
    ax.set(xlabel='Attack acceptance rate',ylabel='Real acceptance rate')
    fig.savefig(str(prefix)+'_roc.png',bbox_inches='tight');plt.close(fig)

ORT_THREADS = 2
def cpu_session(path):
    opts=ort.SessionOptions();opts.intra_op_num_threads=ORT_THREADS;opts.inter_op_num_threads=1
    return ort.InferenceSession(str(path),sess_options=opts,providers=['CPUExecutionProvider'])

def load_frozen_model(kind, directory, mode):
    names={'E1':f'mnv3s_e1_{mode}_v5_3_edge','E3':f'e3_concat_{mode}_v1'}
    stem=names[kind]
    onnx_path=find_exact_artifact(stem+'_best.onnx',directory)
    config=json.loads(Path(find_exact_artifact(stem+'_runtime_config.json',directory)).read_text())
    meta=json.loads(Path(find_exact_artifact(stem+'_best_meta.json',directory)).read_text())
    thresholds=[float(x['calibrated_logit_threshold']) for x in [config,meta]]
    if not np.isfinite(thresholds).all() or abs(thresholds[0]-thresholds[1])>1e-9: raise RuntimeError('Conflicting locked thresholds')
    for x in [config,meta]:
        if not np.isfinite(float(x.get('calibrated_probability_threshold',x.get('predictor_threshold_probability',float('nan'))))) or abs(float(x.get('calibrated_probability_threshold',x.get('predictor_threshold_probability',float('nan'))))-sigmoid(thresholds[0]))>1e-7: raise RuntimeError('Probability threshold conflict')
    if config.get('onnx_sha256') and config['onnx_sha256']!=sha256_file(onnx_path): raise RuntimeError('ONNX hash mismatch')
    if kind=='E3':
        if config['e1_run_mode']!=mode or meta['e1_run_mode']!=mode: raise RuntimeError('Run-mode mismatch')
        if config['init_mode']!='independent': raise RuntimeError('Primary comparison requires independent E3')
        if config['dct_spec']!=DCT_SPEC: raise RuntimeError('DCT contract mismatch')
        if config['normalization_mean']!=CELEBA_MEAN or config['normalization_std']!=CELEBA_STD: raise RuntimeError('RGB contract mismatch')
        if config['split_fingerprints']!=meta['split_fingerprints'] or config['bbox_cache_selected_fingerprint']!=meta['bbox_cache_selected_fingerprint']: raise RuntimeError('Metadata provenance conflict')
    else:
        if meta.get('backbone_name')!='MobileNetV3-Small' or meta.get('sample_seed')!=42: raise RuntimeError('Frozen E1 architecture/seed mismatch')
        if meta.get('train_samples')!= (98873 if mode=='preliminary' else meta.get('train_samples')): raise RuntimeError('E1 mode mismatch')
        if config.get('apply_gamma',config.get('research_apply_gamma',False)): raise RuntimeError('Gamma must be OFF')
        if config.get('mean',config.get('normalization_mean'))!=CELEBA_MEAN or config.get('std',config.get('normalization_std'))!=CELEBA_STD: raise RuntimeError('E1 normalization mismatch')
    if kind=='E1' and (config.get('bbox_expansion_factor')!=1.55 or config.get('model_img_size')!=224 or meta.get('bbox_expansion_factor')!=1.55): raise RuntimeError('Frozen E1 crop/input contract mismatch')
    session=cpu_session(onnx_path)
    ins=session.get_inputs()
    expected=[[3,224,224],[1,224,224]] if kind=='E3' else [[3,224,224]]
    if len(ins)!=len(expected) or any(i.type!='tensor(float)' or i.shape[1:]!=s for i,s in zip(ins,expected)): raise RuntimeError('ONNX input contract mismatch')
    if kind=='E3' and [i.name for i in ins]!=['rgb_input','frequency_map']: raise RuntimeError('E3 input names mismatch')
    if session.get_outputs()[0].shape[-1]!=3: raise RuntimeError('Output contract mismatch')
    return {'session':session,'threshold':thresholds[0],'config':config,'meta':meta,'onnx_path':onnx_path,'sha256':sha256_file(onnx_path),'onnx_sidecar_sha256':{p.name:sha256_file(p) for p in Path(onnx_path).parent.glob(Path(onnx_path).name+'.data*')}}

def infer_frozen(bundle,rgb,freq):
    ins=bundle['session'].get_inputs()
    vals=[rgb.numpy()[None].astype(np.float32),freq.numpy()[None].astype(np.float32)]
    logits=bundle['session'].run(None,{i.name:v for i,v in zip(ins,vals)})[0]
    if logits.shape!=(1,3) or not np.isfinite(logits).all(): raise RuntimeError('Invalid model inference')
    return logits[0]

## Frozen preflight — no split reconstruction or SCRFD rerun

In [ ]:
if E1_RUN_MODE not in {'preliminary','official'}: raise ValueError('Invalid mode')
TRAIN_JSON = os.path.join(DATA_ROOT,'metas/intra_test/train_label.json')
TEST_JSON = os.path.join(DATA_ROOT,'metas/intra_test/test_label.json')
SCRFD_CACHE_PATH=find_exact_artifact(f'celeba_scrfd_bbox_cache_v5_3_{E1_RUN_MODE}_seed42.json',E1_ARTIFACT_DIR)
SPLIT_MANIFEST_PATH=find_exact_artifact(f'celeba_spoof_{E1_RUN_MODE}_v5_3_edge_mnv3_small_seed42.npz',E1_ARTIFACT_DIR)
E1_RUN_CONFIG_PATH=find_exact_artifact(f'mnv3s_e1_{E1_RUN_MODE}_v5_3_edge_run_config.json',E1_ARTIFACT_DIR)
e1_config=json.loads(Path(E1_RUN_CONFIG_PATH).read_text())
if e1_config['run_mode']!=E1_RUN_MODE: raise RuntimeError('Mixed E1 run modes')
manifest=np.load(SPLIT_MANIFEST_PATH,allow_pickle=False)
train_keys,val_keys,test_keys=[manifest[s+'_keys'].astype(str) for s in ['train','val','test']]
verify_disjoint([train_keys,val_keys,test_keys])
requested=manifest['train_keys_requested'].astype(str)
if not set(train_keys)<=set(requested): raise RuntimeError('TRAIN filtering mismatch')
cache_payload=json.loads(Path(SCRFD_CACHE_PATH).read_text());bbox_cache=cache_payload['records'];policy=cache_payload['policy']
if cache_payload['schema_version']!='celeba_scrfd_bbox_cache_v5_3': raise RuntimeError('Cache schema mismatch')
for k,v in {'scrfd_crop_factor':1.55,'celeba_fallback_crop_factor':1.5,'train_min_face_px':48,'det_thresh':0.5,'primary_input_size':[640,640],'fallback_input_sizes':[[480,480],[320,320]]}.items():
    if policy.get(k)!=v: raise RuntimeError(f'Cache policy mismatch: {k}')
selected_keys=list(dict.fromkeys(list(requested)+list(val_keys)+list(test_keys)))
if any(k not in bbox_cache for k in selected_keys): raise RuntimeError('Missing bbox records')
split_fingerprints={s:split_fingerprint(keys) for s,keys in zip(['train','val','test'],[train_keys,val_keys,test_keys])}
cache_fingerprint=selected_cache_fingerprint(selected_keys,bbox_cache)
if split_fingerprints!=e1_config['split_fingerprints']: raise RuntimeError('Split fingerprint mismatch')
if cache_fingerprint!=e1_config['bbox_cache_selected_fingerprint']: raise RuntimeError('Cache fingerprint mismatch')
train_meta=json.loads(Path(TRAIN_JSON).read_text());test_meta=json.loads(Path(TEST_JSON).read_text())
counts={s:class_counts(keys,meta) for s,keys,meta in [('train',train_keys,train_meta),('val',val_keys,train_meta),('test',test_keys,test_meta)]}
# Authoritative preliminary counts: executed frozen E1/E2 notebook output.
expected_counts = e1_config.get('class_counts')
if expected_counts is None and E1_RUN_MODE=='preliminary':
    expected_counts={'train':{0:32389,1:45761,2:20723},'val':{0:4980,1:6957,2:3179},'test':{0:2966,1:5284,2:1750}}
if expected_counts is None:
    if not FROZEN_CLASS_COUNTS_PATH: raise RuntimeError('Official frozen class-count reference missing; mount exported E1 counts, never derive a new reference from current metadata')
    expected_counts=json.loads(Path(FROZEN_CLASS_COUNTS_PATH).read_text())
expected_counts={s:{int(k):int(v) for k,v in c.items()} for s,c in expected_counts.items()}
if counts!=expected_counts: raise RuntimeError('Frozen class counts mismatch')
# NPZ stores keys, not class counts. Cross-check frozen cache attack codes too.
for keys,meta in [(train_keys,train_meta),(val_keys,train_meta),(test_keys,test_meta)]:
    for k in keys:
        rec=bbox_cache[k]
        if rec['status']=='INVALID' or rec['crop_factor']!=(1.55 if rec['bbox_source'].startswith('SCRFD_') else 1.50): raise RuntimeError('Cached crop mismatch')
        if 'attack_code' in rec and int(rec['attack_code'])!=int(meta[k][40]): raise RuntimeError('Label mismatch')
if len(train_keys)!=e1_config['preprocess_stats']['train_retained'] or len(val_keys)!=e1_config['val_actual_size'] or len(test_keys)!=e1_config['test_sample_size']: raise RuntimeError('Split counts mismatch')
print('=== Frozen E1 preflight PASS ===',counts,split_fingerprints,cache_fingerprint)

## One crop and one augmentation feed both branches

In [ ]:
train_augment=A.Compose([A.HorizontalFlip(p=.5),A.RandomBrightnessContrast(brightness_limit=.10,contrast_limit=.10,p=.30)])
# Albumentations owns a generator in recent versions; explicitly seed it as well.
if hasattr(train_augment,'set_random_seed'): train_augment.set_random_seed(42)
class SharedCropDataset(Dataset):
    def __init__(self,keys,meta,training=False): self.keys=list(keys);self.meta=meta;self.training=training
    def __len__(self): return len(self.keys)
    def __getitem__(self,index):
        k=self.keys[index]
        crop=read_cached_face_crop_rgb(os.path.join(DATA_ROOT,k),bbox_cache[k],bbox_jitter=self.training,bbox_jitter_p=.20)
        if self.training: crop=train_augment(image=crop)['image']
        rgb,freq=branch_inputs(crop)
        return rgb,freq,torch.tensor(label_of(self.meta[k]),dtype=torch.long)

def seed_worker(worker_id):
    seed=torch.initial_seed()%2**32;np.random.seed(seed);random.seed(seed)
    if hasattr(train_augment,'set_random_seed'): train_augment.set_random_seed(seed)
loader_generator=torch.Generator().manual_seed(42)
loader_args=dict(batch_size=128,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),worker_init_fn=seed_worker)
train_loader=DataLoader(SharedCropDataset(train_keys,train_meta,True),shuffle=True,generator=loader_generator,drop_last=len(train_keys)%128==1,**loader_args)
val_loader=DataLoader(SharedCropDataset(val_keys,train_meta),shuffle=False,**loader_args)
# No Test loader exists in the training notebook.

## Exact E2 FrequencyBranch; E1 spatial representation; CONCAT only

In [ ]:
class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                in_ch,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_ch,
                bias=False,
            ),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class FrequencyBranch(nn.Module):
    def __init__(self, out_dim=FREQ_DIM):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            DepthwiseSeparableBlock(32, 32, stride=2),
            DepthwiseSeparableBlock(32, 64, stride=2),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.proj(self.features(x))

class E3ConcatPAD(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        backbone=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None)
        self.features=backbone.features
        self.avgpool=backbone.avgpool
        self.spatial_projection=nn.Sequential(nn.Linear(576,256),nn.BatchNorm1d(256),nn.Hardswish(inplace=True),nn.Dropout(.2))
        self.frequency_branch=FrequencyBranch(64)
        self.classifier=nn.Sequential(nn.Linear(320,128),nn.Hardswish(inplace=True),nn.Dropout(.2),nn.Linear(128,3))
    def forward(self,rgb_input,frequency_map):
        if not torch.jit.is_tracing() and rgb_input.shape[0]!=frequency_map.shape[0]: raise ValueError('Input batch mismatch')
        spatial=self.spatial_projection(torch.flatten(self.avgpool(self.features(rgb_input)),1))
        return self.classifier(torch.cat([spatial,self.frequency_branch(frequency_map)],dim=1))

In [ ]:
def calibrate_threshold_min_acer(pad_scores, binary_labels):
    """
    Score convention: d = real_logit - logsumexp(spoof_logits).
    Predict Real when d >= threshold, Spoof when d < threshold.
    Threshold is chosen on Validation ONLY by minimum ACER.
    Tie-break: lower APCER, then lower BPCER.
    """
    d = np.asarray(pad_scores, dtype=np.float64)
    y = np.asarray(binary_labels, dtype=np.int64)
    real_scores = np.sort(d[y == 0])
    spoof_scores = np.sort(d[y == 1])
    if len(real_scores) == 0 or len(spoof_scores) == 0:
        raise ValueError("Validation set must contain both Real and Spoof samples")
    if not np.all(np.isfinite(d)):
        raise ValueError("Non-finite PAD scores detected during validation")

    unique_scores = np.unique(d)
    thresholds = np.concatenate((
        [np.nextafter(unique_scores[0], -np.inf)],
        unique_scores,
        [np.nextafter(unique_scores[-1], np.inf)],
        [PREDICTOR_MIN_LOGIT_THRESHOLD, PREDICTOR_MAX_LOGIT_THRESHOLD],
    ))
    # Predictor clamps probability threshold to [1e-6, 1-1e-6]. Keep calibration
    # inside the exactly representable logit range so deployment decisions are identical.
    thresholds = np.unique(np.clip(
        thresholds, PREDICTOR_MIN_LOGIT_THRESHOLD, PREDICTOR_MAX_LOGIT_THRESHOLD
    ))

    # BPCER = Real -> Spoof = P(d < t | real)
    bpcer = np.searchsorted(real_scores, thresholds, side="left") / len(real_scores)
    # APCER = Spoof -> Real = P(d >= t | spoof)
    apcer = 1.0 - np.searchsorted(spoof_scores, thresholds, side="left") / len(spoof_scores)
    acer = 0.5 * (apcer + bpcer)

    # np.lexsort uses the LAST key as primary: ACER -> APCER -> BPCER.
    best_idx = np.lexsort((bpcer, apcer, acer))[0]
    logit_threshold = float(thresholds[best_idx])
    return {
        "logit_threshold": logit_threshold,
        "probability_threshold": probability_from_logit_threshold(logit_threshold),
        "APCER": float(apcer[best_idx]),
        "BPCER": float(bpcer[best_idx]),
        "ACER": float(acer[best_idx]),
    }

def probability_from_logit_threshold(t): return sigmoid(t)
PREDICTOR_MIN_LOGIT_THRESHOLD=float(np.log(1e-6/(1-1e-6)))
PREDICTOR_MAX_LOGIT_THRESHOLD=-PREDICTOR_MIN_LOGIT_THRESHOLD

## Train and select using Validation ACER only

In [ ]:
assert INIT_MODE=='independent', 'Only independent initialization is implemented in primary v1'
RUN_NAME=f'e3_concat_{E1_RUN_MODE}_v1'
OUT_DIR=Path('/kaggle/working')/RUN_NAME;OUT_DIR.mkdir(parents=True,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=E3ConcatPAD(pretrained=not bool(CHECKPOINT_INPUT_PATH)).to(device)
params={'spatial':sum(p.numel() for m in [model.features,model.spatial_projection] for p in m.parameters()),'frequency':sum(p.numel() for p in model.frequency_branch.parameters()),'fusion':sum(p.numel() for p in model.classifier.parameters())}
params['total']=sum(params.values())
config={'protocol_version':'e3_concat_v1','e1_run_mode':E1_RUN_MODE,'init_mode':INIT_MODE,'seed':42,'split_fingerprints':split_fingerprints,'bbox_cache_selected_fingerprint':cache_fingerprint,'class_counts':counts,
 'manifest_sha256':sha256_file(SPLIT_MANIFEST_PATH),'cache_file_sha256':sha256_file(SCRFD_CACHE_PATH),'dct_spec':DCT_SPEC,'normalization_mean':CELEBA_MEAN,'normalization_std':CELEBA_STD,'gamma':False,
 'architecture':'MobileNetV3-Small 576->256 + exact E2 64; concat 320->128->3','parameter_counts':params,
 'training':{'optimizer':'AdamW','backbone_lr':1e-5,'new_layers_lr':1e-4,'weight_decay':1e-4,'batch_size':128,'epochs':24,'warmup_epochs':2,'warmup_start_factor':.2,'min_lr_factor':.1,'early_stop_patience':4,'early_stop_min_delta':1e-4,'early_stop_min_epochs':10,'grad_clip':5.,'label_smoothing':.1,'class_weights':[1,1,1]},
 'augmentation':e1_config['augmentation'],'strict_determinism':False,'cudnn_benchmark':True,'source_notebook_hashes':SOURCE_NOTEBOOK_HASHES,
 'environment':{'python':sys.version,'torch':torch.__version__,'opencv':cv2.__version__,'albumentations':A.__version__,'torchvision':__import__('torchvision').__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}}
optimizer=torch.optim.AdamW([{'params':model.features.parameters(),'lr':1e-5},{'params':list(model.spatial_projection.parameters())+list(model.frequency_branch.parameters())+list(model.classifier.parameters()),'lr':1e-4}],weight_decay=1e-4)
EPOCHS=24;WARMUP_EPOCHS=2;WARMUP_START_FACTOR=.2;MIN_LR_FACTOR=.1
def lr_multiplier(epoch_index: int) -> float:
    epoch_index = int(epoch_index)

    if WARMUP_EPOCHS > 0 and epoch_index < WARMUP_EPOCHS:
        return WARMUP_START_FACTOR + (
            1.0 - WARMUP_START_FACTOR
        ) * (epoch_index / WARMUP_EPOCHS)

    cosine_epochs = max(1, EPOCHS - WARMUP_EPOCHS)
    denom = max(1, cosine_epochs - 1)
    progress = min(
        1.0,
        max(0.0, (epoch_index - WARMUP_EPOCHS) / denom),
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_multiplier)
criterion=nn.CrossEntropyLoss(weight=torch.ones(3,device=device),label_smoothing=.1)
scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
best_acer=float('inf');best_apcer=float('inf');bad_epochs=0;history=[];start_epoch=1;best_state=None
# Use only your own trusted PyTorch checkpoint: it includes optimizer/RNG state.
if (OUT_DIR/(RUN_NAME+'_best.pth')).exists(): raise RuntimeError('Existing output: archive it or use a clean Kaggle session')
if CHECKPOINT_INPUT_PATH:
    checkpoint=torch.load(CHECKPOINT_INPUT_PATH,map_location='cpu',weights_only=False)
    if checkpoint['config']!=config: raise RuntimeError('Resume config/environment mismatch; do not alter the frozen protocol')
    model.load_state_dict(checkpoint['model_state_dict']);optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler']);scaler.load_state_dict(checkpoint['scaler'])
    history=checkpoint['history'];best_acer=checkpoint['best_acer'];bad_epochs=checkpoint['bad_epochs']
    best_meta=checkpoint['best_meta'];best_state=checkpoint['best_state'];best_apcer=best_meta['validation_metrics']['APCER']
    start_epoch=checkpoint['epoch']+1
    torch.save(best_state,OUT_DIR/(RUN_NAME+'_best.pth'));save_json(OUT_DIR/(RUN_NAME+'_best_meta.json'),best_meta)
    random.setstate(checkpoint['python_rng']);np.random.set_state(checkpoint['numpy_rng']);torch.set_rng_state(checkpoint['torch_rng'])
    if torch.cuda.is_available(): torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
    loader_generator.set_state(checkpoint['loader_rng'])
    shutil.copyfile(CHECKPOINT_INPUT_PATH,OUT_DIR/(RUN_NAME+'_checkpoint_last.pth'))
    print('Resuming after epoch',checkpoint['epoch'])
save_json(OUT_DIR/(RUN_NAME+'_run_config.json'),config)
# Preserve the stop condition from the interrupted run.
if start_epoch>10 and bad_epochs>=4: start_epoch=25
for epoch in range(start_epoch,25):
    model.train();loss_sum=0.;n=0
    for rgb,freq,y in tqdm(train_loader,desc=f'Epoch {epoch}'):
        rgb,freq,y=[t.to(device,non_blocking=True) for t in [rgb,freq,y]]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type,enabled=device.type=='cuda'):
            loss=criterion(model(rgb,freq),y)
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),5.)
        scaler.step(optimizer);scaler.update();loss_sum+=loss.item()*len(y);n+=len(y)
    model.eval();all_y=[];all_logits=[]
    with torch.no_grad():
        for rgb,freq,y in tqdm(val_loader,desc='Validation'):
            # FP32 Validation locks the same score representation used by ONNX.
            logits=model(rgb.to(device),freq.to(device)).cpu().numpy()
            all_y.extend(y.numpy());all_logits.append(logits)
    d=binary_pad_score_from_logits(np.concatenate(all_logits))
    cal=calibrate_threshold_min_acer(d,(np.asarray(all_y)!=0).astype(int))
    val=pad_metrics(all_y,d,cal['logit_threshold'])
    row={'epoch':epoch,'train_loss':loss_sum/n,'validation':val,'logit_threshold':cal['logit_threshold'],'lrs':[g['lr'] for g in optimizer.param_groups]}
    history.append(row);print(row)
    previous_best=best_acer
    is_better=val['ACER']<best_acer-1e-12 or (abs(val['ACER']-best_acer)<=1e-12 and val['APCER']<best_apcer)
    if is_better:
        best_acer=val['ACER'];best_apcer=val['APCER']
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        torch.save(best_state,OUT_DIR/(RUN_NAME+'_best.pth'))
        best_meta={**config,'best_epoch':epoch,'validation_metrics':val,'calibrated_logit_threshold':cal['logit_threshold'],'calibrated_probability_threshold':cal['probability_threshold'],'threshold_origin':'CelebA-Spoof Validation'}
        save_json(OUT_DIR/(RUN_NAME+'_best_meta.json'),best_meta)
    bad_epochs=0 if val['ACER']<previous_best-1e-4 else bad_epochs+1
    scheduler.step()
    torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'config':config,'history':history,'best_acer':best_acer,'bad_epochs':bad_epochs,'best_state':best_state,'best_meta':best_meta,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],'loader_rng':loader_generator.get_state()},OUT_DIR/(RUN_NAME+'_checkpoint_last.pth'))
    save_json(OUT_DIR/(RUN_NAME+'_training_history.json'),history)
    if epoch>=10 and bad_epochs>=4: break
save_json(OUT_DIR/(RUN_NAME+'_training_history.json'),history)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot([h['epoch'] for h in history],[h['train_loss'] for h in history]);axes[0].set_title('Train loss')
axes[1].plot([h['epoch'] for h in history],[h['validation']['ACER'] for h in history]);axes[1].set_title('Validation ACER')
fig.savefig(OUT_DIR/(RUN_NAME+'_training_history.png'),bbox_inches='tight');plt.close(fig)

## Two-input ONNX export; external DCT; current app needs future integration

In [ ]:
import onnx
model=model.cpu().eval();model.load_state_dict(torch.load(OUT_DIR/(RUN_NAME+'_best.pth'),map_location='cpu',weights_only=True))
onnx_path=OUT_DIR/(RUN_NAME+'_best.onnx')
dummy_rgb=torch.randn(2,3,224,224);dummy_freq=torch.randn(2,1,224,224)
torch.onnx.export(model,(dummy_rgb,dummy_freq),str(onnx_path),input_names=['rgb_input','frequency_map'],output_names=['logits'],dynamic_axes={'rgb_input':{0:'batch'},'frequency_map':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)
onnx.checker.check_model(str(onnx_path));session=cpu_session(onnx_path)
parity=[]
for batch in [1,2]:
    r=torch.randn(batch,3,224,224);f=torch.randn(batch,1,224,224)
    with torch.no_grad(): expected=model(r,f).numpy()
    actual=session.run(None,{'rgb_input':r.numpy(),'frequency_map':f.numpy()})[0]
    delta=float(np.max(np.abs(expected-actual)));parity.append(delta)
    if delta>=1e-4: raise RuntimeError(f'ONNX parity failed: {delta}')
# Also check an actual unaugmented Validation crop.
k=str(val_keys[0]);crop=read_cached_face_crop_rgb(os.path.join(DATA_ROOT,k),bbox_cache[k]);r,f=branch_inputs(crop)
with torch.no_grad(): expected=model(r[None],f[None]).numpy()
actual=session.run(None,{'rgb_input':r.numpy()[None],'frequency_map':f.numpy()[None]})[0]
delta=float(np.max(np.abs(expected-actual)));parity.append(delta)
if delta>=1e-4: raise RuntimeError('Real-crop ONNX parity failed')
runtime={**best_meta,'onnx_sha256':sha256_file(onnx_path),'onnx_inputs':[{'name':'rgb_input','shape':['N',3,224,224]},{'name':'frequency_map','shape':['N',1,224,224]}],'onnx_output':'logits','external_dct_required':True,'current_app_compatible':False,'pad_score':'real_logit - logsumexp(spoof_logits)','decision':'Real iff d >= calibrated_logit_threshold','onnx_parity_max_abs':max(parity)}
save_json(OUT_DIR/(RUN_NAME+'_runtime_config.json'),runtime)
print('Frozen export PASS',runtime['onnx_parity_max_abs'])

## Host efficiency diagnostics and download

In [ ]:
# Matched host diagnostics; this is not Raspberry Pi or camera latency.
WARMUP=10;TIMED_SAMPLES=100

def timing_stats(values): return {'mean_ms':float(np.mean(values)),'p50_ms':float(np.percentile(values,50)),'p95_ms':float(np.percentile(values,95))}

def benchmark_model(bundle,image,bbox,factor,with_frequency):
    process=psutil.Process();base=process.memory_info().rss;peak=base
    timings={k:[] for k in ['crop_resize_spatial','luminance_dct','model_only','total_pad']}
    for i in range(WARMUP+TIMED_SAMPLES):
        t0=time.perf_counter();crop=cv2.cvtColor(runtime_crop_bgr(image,bbox,factor),cv2.COLOR_BGR2RGB)
        rgb=runtime_preprocess_rgb(crop,224,CELEBA_MEAN,CELEBA_STD).numpy()[None];t1=time.perf_counter()
        freq=rgb_to_frequency_tensor(crop).numpy()[None] if with_frequency else None;t2=time.perf_counter()
        ins=bundle['session'].get_inputs();feed={ins[0].name:rgb}
        if with_frequency: feed[ins[1].name]=freq
        bundle['session'].run(None,feed);t3=time.perf_counter()
        peak=max(peak,process.memory_info().rss)
        if i>=WARMUP:
            for key,value in zip(timings,[(t1-t0)*1000,(t2-t1)*1000,(t3-t2)*1000,(t3-t0)*1000]): timings[key].append(value)
    return {'timing':{k:timing_stats(v) for k,v in timings.items()},'rss_before_bytes':base,'sampled_peak_process_rss_bytes':peak,'memory_method':'psutil RSS sampled after each iteration; whole notebook process, not isolated model allocation', 'onnx_fp32_bytes':Path(bundle['onnx_path']).stat().st_size+sum(p.stat().st_size for p in Path(bundle['onnx_path']).parent.glob(Path(bundle['onnx_path']).name+'.data*'))}

settings={'provider':'CPUExecutionProvider','intra_op_threads':ORT_THREADS,'inter_op_threads':1,'batch_size':1,'warmup':WARMUP,'timed_samples':TIMED_SAMPLES,'host':platform.platform(),'processor':platform.processor(),'includes_detector':False,'includes_io':False,'notes':'Host diagnostic; no target-device/camera claim. Same actual Validation image/crop for both models.'}
image=cv2.imread(os.path.join(DATA_ROOT,k));rec=bbox_cache[k]
e3_bundle=load_frozen_model('E3',str(OUT_DIR),E1_RUN_MODE)
efficiency={'settings':settings,'parameters':params,'E3':benchmark_model(e3_bundle,image,rec['bbox_xyxy'],rec['crop_factor'],True)}
try:
    e1_bundle=load_frozen_model('E1',E1_ARTIFACT_DIR,E1_RUN_MODE)
except FileNotFoundError:
    print('E1 model absent: paired efficiency comparison pending. E1 weights are not required for E3 training.')
else:
    efficiency['E1']=benchmark_model(e1_bundle,image,rec['bbox_xyxy'],rec['crop_factor'],False)
save_json(OUT_DIR/(RUN_NAME+'_efficiency.json'),efficiency)
shutil.make_archive(str(OUT_DIR),'zip',OUT_DIR)
print('Download:',str(OUT_DIR)+'.zip')